# MFCC + SVM

**Модель:** SVM (RBF) на статистиках MFCC (mean + std + delta_mean + delta_std)

In [38]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score
from collections import Counter

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils, train_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds
from shared.results_utils import save_result_csv

In [39]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_test_split()

print(f"Train+Val: {len(paths_trainval)} good: {np.sum(labels_trainval==0)}, bad: {np.sum(labels_trainval==1)}")
print(f"Test:      {len(paths_test)}  good: {np.sum(labels_test==0)},  bad: {np.sum(labels_test==1)}")

Train+Val: 2356 good: 1594, bad: 762
Test:      416  good: 281,  bad: 135


## 5-Fold CV (для подбора гиперпараметров)

In [40]:
extractor = data_utils.extract_mfcc_stats

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
f1_bad_scorer = make_scorer(f1_score, pos_label=config.CLASS_BAD, average="binary")

param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
}


print("Извлечение признаков (train+val)...")
X_all   = build_feature_matrix(paths_trainval, extractor)
path_to_idx = {p: i for i, p in enumerate(paths_trainval)}

print("Извлечение признаков (test)...")
X_test_raw = build_feature_matrix(paths_test, extractor)

fold_results = []
all_best_params = []
t0 = time.perf_counter()

for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
        get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

    idx_tr  = [path_to_idx[p] for p in paths_tr]
    idx_val = [path_to_idx[p] for p in paths_val]
    X_tr  = np.hstack([X_all[idx_tr],  letters_tr])
    X_val = np.hstack([X_all[idx_val], letters_val])

    grid = GridSearchCV(pipeline, param_grid, cv=3,
                        scoring=f1_bad_scorer, refit=True, n_jobs=-1)
    grid.fit(X_tr, labels_tr)
    clf = grid.best_estimator_
    all_best_params.append(grid.best_params_)
    y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
    thr = find_optimal_threshold(labels_val, y_proba_val)
    metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=False)
    print(f"Фолд {fold+1}/{config.CV_N_SPLITS}, best_params={grid.best_params_}, threshold={thr}")
    fold_results.append(metrics)

train_time_sec = time.perf_counter() - t0
cv_agg = evaluate_cv_folds(fold_results)
print_cv_summary(cv_agg)

Извлечение признаков (train+val)...
Извлечение признаков (test)...
Фолд 1/5, best_params={'clf__C': 1.0, 'clf__gamma': 0.01}, threshold=0.35
Фолд 2/5, best_params={'clf__C': 50.0, 'clf__gamma': 0.001}, threshold=0.3
Фолд 3/5, best_params={'clf__C': 10.0, 'clf__gamma': 0.001}, threshold=0.29
Фолд 4/5, best_params={'clf__C': 1.0, 'clf__gamma': 0.01}, threshold=0.28
Фолд 5/5, best_params={'clf__C': 1.0, 'clf__gamma': 0.01}, threshold=0.18
accuracy          0.7042±0.0349
f1_macro          0.6931±0.0308
f1_bad            0.6386±0.0212
roc_auc           0.7906±0.0264
precision_bad     0.5318±0.0332
recall_bad        0.8058±0.0557
Средний threshold: 0.28


## Финальная модель: обучение на train+val, оценка на test

In [41]:
X_trainval = np.hstack([X_all,      letters_trainval])
X_test     = np.hstack([X_test_raw, letters_test])


best_params = dict(
    Counter(tuple(sorted(p.items())) for p in all_best_params).most_common(1)[0][0]
)
print(f"Лучшие параметры из CV: {best_params}")
pipeline.set_params(**best_params)
pipeline.fit(X_trainval, labels_trainval)

cv_threshold = cv_agg["threshold_mean"]
y_proba_test = pipeline.predict_proba(X_test)[:, config.CLASS_BAD]
test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)
pd.DataFrame({
    "path":    paths_test,
    "y_true":  labels_test,
    "y_pred":  (y_proba_test >= cv_threshold).astype(int),
    "y_proba": y_proba_test,
}).to_csv(exp_dir / "test_predictions.csv", index=False)

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_mfcc_svm",
    experiment_name="MFCC + SVM (baseline)",
    f1_bad=test_metrics["f1_bad"],
    f1_macro=test_metrics["f1_macro"],
    roc_auc=test_metrics["roc_auc"],
    accuracy=test_metrics["accuracy"],
    precision_bad=test_metrics["precision_bad"],
    recall_bad=test_metrics["recall_bad"],
    threshold=test_metrics["threshold"],
    embed_dim=80,
    embed_dim_note="MFCC 20 коэф. * 4 статистики (mean, std, delta_mean, delta_std)",
    notes=f"5-fold CV + test | best_params={best_params} | cv_thr={cv_threshold:.2f}",
    train_time_sec=train_time_sec,
)

Лучшие параметры из CV: {'clf__C': 1.0, 'clf__gamma': 0.01}
              precision    recall  f1-score   support

        good       0.83      0.72      0.77       281
         bad       0.54      0.70      0.61       135

    accuracy                           0.71       416
   macro avg       0.69      0.71      0.69       416
weighted avg       0.74      0.71      0.72       416

Threshold : 0.28
Accuracy  : 0.7115
F1-macro  : 0.6907
F1-bad    : 0.6104
ROC-AUC   : 0.7840


PosixPath('/Users/dk/Desktop/ВШЭ/ВКР/HSE_VKR_DetectingSpeechDefects/experiments/01_baselines/exp_mfcc_svm/result.csv')